In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from collections import Counter
import math
import string
import numpy as np
import os

# Задание 1 (Подготовка и шифрование)

In [ ]:
ENGLISH_ALPHABET = 'abcdefghijklmnopqrstuvwxyz'
RUSSIAN_ALPHABET = 'абвгдеёжзийклмнопрстуфхцчшщъыьэюя'

RUSSIAN_ALPHABET_SIZE = len(RUSSIAN_ALPHABET)

class VigenereCipher:
    def __init__(self):
        self.alphabet = ENGLISH_ALPHABET + RUSSIAN_ALPHABET
        self.alphabet_size = len(self.alphabet)
        
        # Создаем таблицы для быстрого доступа к индексам
        self.char_to_index = {char: idx for idx, char in enumerate(self.alphabet)}
        self.index_to_char = {idx: char for idx, char in enumerate(self.alphabet)}
    
    def _get_shift(self, key_char):
        """Получить сдвиг для символа ключа"""
        return self.char_to_index.get(key_char.lower(), 0)
    
    def encrypt(self, text, key):
        """Шифрование текста"""
        encrypted = []
        key = key.lower()
        key_length = len(key)
        key_index = 0
        
        for char in text:
            if char.lower() in self.char_to_index:
                # Определяем сдвиг
                shift = self._get_shift(key[key_index % key_length])
                
                # Шифруем символ
                char_idx = self.char_to_index[char.lower()]
                encrypted_idx = (char_idx + shift) % self.alphabet_size
                encrypted_char = self.index_to_char[encrypted_idx]
                
                # Сохраняем регистр
                if char.isupper():
                    encrypted_char = encrypted_char.upper()
                
                encrypted.append(encrypted_char)
                key_index += 1
            else:
                # Оставляем символы не из алфавита как есть
                encrypted.append(char)
        
        return ''.join(encrypted)
    
    def decrypt(self, text, key):
        """Расшифрование текста"""
        decrypted = []
        key = key.lower()
        key_length = len(key)
        key_index = 0
        
        for char in text:
            if char.lower() in self.char_to_index:
                # Определяем сдвиг
                shift = self._get_shift(key[key_index % key_length])
                
                # Расшифровываем символ
                char_idx = self.char_to_index[char.lower()]
                decrypted_idx = (char_idx - shift) % self.alphabet_size
                decrypted_char = self.index_to_char[decrypted_idx]
                
                # Сохраняем регистр
                if char.isupper():
                    decrypted_char = decrypted_char.upper()
                
                decrypted.append(decrypted_char)
                key_index += 1
            else:
                # Оставляем символы не из алфавита как есть
                decrypted.append(char)
        
        return ''.join(decrypted)

In [ ]:
def read_text_from_file(filename):
    """Чтение текста из файла"""
    print(os.getcwd())
    try:
        with open(filename, 'r', encoding='utf-8') as file:
            return file.read()
    except FileNotFoundError:
        print(f"Файл {filename} не найден")
        return None

def write_text_to_file(filename, text):
    """Запись текста в файл"""
    with open(filename, 'w', encoding='utf-8') as file:
        file.write(text)

In [ ]:
def analyze_text_frequency_separated(text, title):
    """Анализ частотности символов с разделением по алфавитам"""
    # Разделяем символы по алфавитам
    english_chars = [char.lower() for char in text if char.lower() in ENGLISH_ALPHABET]
    russian_chars = [char.lower() for char in text if char.lower() in RUSSIAN_ALPHABET]
    
    # Анализ для английских символов
    if english_chars:
        eng_counter = Counter(english_chars)
        total_eng = len(english_chars)
        
        eng_freq_data = []
        for char in ENGLISH_ALPHABET:
            count = eng_counter.get(char, 0)
            eng_freq_data.append({
                'character': char,
                'count': count,
                'frequency': count / total_eng * 100 if total_eng > 0 else 0,
                'alphabet': 'Английский'
            })
        
        df_eng = pd.DataFrame(eng_freq_data)
        
        fig_eng = px.bar(df_eng, x='character', y='frequency', 
                        title=f'Частотный анализ (Английский): {title}',
                        labels={'character': 'Символ', 'frequency': 'Частота (%)'})
        fig_eng.show()
    else:
        print("Нет английских символов для анализа")
        df_eng = None
    
    # Анализ для русских символов
    if russian_chars:
        rus_counter = Counter(russian_chars)
        total_rus = len(russian_chars)
        
        rus_freq_data = []
        for char in RUSSIAN_ALPHABET:
            count = rus_counter.get(char, 0)
            rus_freq_data.append({
                'character': char,
                'count': count,
                'frequency': count / total_rus * 100 if total_rus > 0 else 0,
                'alphabet': 'Русский'
            })
        
        df_rus = pd.DataFrame(rus_freq_data)
        
        fig_rus = px.bar(df_rus, x='character', y='frequency', 
                        title=f'Частотный анализ (Русский): {title}',
                        labels={'character': 'Символ', 'frequency': 'Частота (%)'})
        fig_rus.show()
    else:
        print("Нет русских символов для анализа")
        df_rus = None
    
    return df_eng, df_rus

In [ ]:
def calculate_text_statistics(text, alphabet):    
    chars = [char.lower() for char in text if char.lower() in alphabet]
    if chars:
        counter = Counter(chars)
        total = len(chars)
        
        stats = []
        for char in alphabet:
            count = counter.get(char, 0)
            frequency = count / total * 100 if total > 0 else 0
            stats.append({
                'character': char,
                'count': count,
                'frequency': frequency
            })
    
    return pd.DataFrame(stats)

def plot_three_statistics_comparison(df1, df2, df3, labels):
  
    if len(labels) != 3:
        raise ValueError("labels должен содержать 3 элемента")
    
    # Сортируем по алфавиту для consistency
    df1_sorted = df1.sort_values('character')
    df2_sorted = df2.sort_values('character')
    df3_sorted = df3.sort_values('character')
    
    fig = go.Figure()
    
    fig.add_trace(go.Bar(
        name=labels[0], 
        x=df1_sorted['character'], 
        y=df1_sorted['frequency'],
        marker_color='#1f77b4'  # синий
    ))
    fig.add_trace(go.Bar(
        name=labels[1], 
        x=df2_sorted['character'], 
        y=df2_sorted['frequency'],
        marker_color='#ff7f0e'  # оранжевый
    ))
    fig.add_trace(go.Bar(
        name=labels[2], 
        x=df3_sorted['character'], 
        y=df3_sorted['frequency'],
        marker_color='#2ca02c'  # зеленый
    ))
    
    fig.update_layout(
        title=f'Сравнение частот',
        xaxis_title='Символ',
        yaxis_title='Частота (%)',
        barmode='group',
        height=600,
        width=1200
    )
        
    fig.show()

In [ ]:
# Обновленный анализ для Задания 1 с разделением по алфавитам
print("=== ЗАДАНИЕ 1 (с разделением по алфавитам) ===")

# Читаем текст из файла
text = read_text_from_file('../../big.txt')
if text is None:
    text = generate_large_text()

print(f"Длина исходного текста: {len(text)} символов")

cipher = VigenereCipher()

# Шифрование коротким ключом
short_key = "code"
encrypted_short = cipher.encrypt(text, short_key)

# Шифрование длинным ключом
long_key = "cryptographysecurity"
encrypted_long = cipher.encrypt(text, long_key)

# Визуальный анализ
sample_text = text[:500] if len(text) > 500 else text
sample_short = encrypted_short[:500] if len(encrypted_short) > 500 else encrypted_short
sample_long = encrypted_long[:500] if len(encrypted_long) > 500 else encrypted_long
print("\n--- Визуальный анализ ---")
print("Исходный текст (первые 500 символов):")
print(sample_text)
print("Зашифрованный коротким ключом 'code' (первые 500 символов):")
print(sample_short)
print("Зашифрованный длинным ключом 'cryptographysecurity' (первые 500 символов):")
print(sample_long)

# Статистический анализ исходного текста с разделением
print("\n--- Статистический анализ исходного текста ---")

df_original_eng = calculate_text_statistics(text, ENGLISH_ALPHABET)
df_original_rus = calculate_text_statistics(text, RUSSIAN_ALPHABET)

df_short_eng = calculate_text_statistics(encrypted_short, ENGLISH_ALPHABET)
df_short_rus = calculate_text_statistics(encrypted_short, RUSSIAN_ALPHABET)

df_long_eng = calculate_text_statistics(encrypted_long, ENGLISH_ALPHABET)
df_long_rus = calculate_text_statistics(encrypted_long, RUSSIAN_ALPHABET)

# Строим графики сравнения для английского алфавита
print("\n--- Сравнительный анализ: Английский алфавит ---")
if df_original_eng is not None and df_short_eng is not None and df_long_eng is not None:
    plot_three_statistics_comparison(
        df_original_eng, 
        df_short_eng, 
        df_long_eng, 
        ['Исходный текст', 'Короткий ключ', 'Длинный ключ']
    )

# Строим графики сравнения для русского алфавита
print("\n--- Сравнительный анализ: Русский алфавит ---")
if df_original_rus is not None and df_short_rus is not None and df_long_rus is not None:
    plot_three_statistics_comparison(
        df_original_rus, 
        df_short_rus, 
        df_long_rus, 
        ['Исходный текст', 'Короткий ключ', 'Длинный ключ']
    )

# Задание 2 (Криптоанализ - определение длины ключа)

In [ ]:
import math
from collections import Counter, defaultdict
import itertools

In [ ]:
def kasiski_examination(ciphertext, min_ngram=3, max_ngram=6, key_count=5, verbose=False):
    """
    Алгоритм Казиски для поиска повторяющихся n-грамм и определения длины ключа
    """
    if verbose:
        print("=== АЛГОРИТМ КАЗИСКИ ===")
    
    # Удаляем пробелы и приводим к верхнему регистру для анализа
    clean_text = ''.join([ch for ch in ciphertext if ch.isalpha()]).upper()
    
    # Ищем повторяющиеся n-граммы
    ngram_positions = defaultdict(list)
    
    for n in range(min_ngram, max_ngram + 1):
        for i in range(len(clean_text) - n + 1):
            ngram = clean_text[i:i+n]
            ngram_positions[ngram].append(i)
    
    # Оставляем только n-граммы, которые встречаются хотя бы 2 раза
    repeating_ngrams = {ngram: positions for ngram, positions in ngram_positions.items() 
                       if len(positions) >= 2}
    
    if verbose:
        print(f"Найдено повторяющихся n-грамм (длина {min_ngram}-{max_ngram}): {len(repeating_ngrams)}")
    
    # Вычисляем расстояния между повторениями
    distances = []
    for ngram, positions in repeating_ngrams.items():
        for i in range(len(positions)):
            for j in range(i + 1, len(positions)):
                distance = positions[j] - positions[i]
                distances.append(distance)
                if verbose and len(repeating_ngrams) <= 10:  # Выводим только если немного n-грамм
                    print(f"n-грамма '{ngram}': позиции {positions[i]}, {positions[j]}, расстояние = {distance}")
    
    if not distances:
        if verbose:
            print("Не найдено повторяющихся n-грамм достаточной длины")
        return []
    
    # Вычисляем НОД для всех расстояний
    def gcd_of_list(numbers):
        if not numbers:
            return 0
        result = numbers[0]
        for num in numbers[1:]:
            result = math.gcd(result, num)
        return result
    
    # Находим все возможные делители
    all_divisors = set()
    for dist in distances:
        for i in range(2, min(50, dist) + 1):  # Ограничиваем поиск делителей
            if dist % i == 0:
                all_divisors.add(i)
    
    # Подсчитываем частоту делителей
    divisor_counts = Counter()
    for dist in distances:
        for divisor in all_divisors:
            if dist % divisor == 0:
                divisor_counts[divisor] += 1
    
    if verbose:
        print(f"\nСтатистика делителей расстояний:")
        for divisor, count in divisor_counts.most_common(10):
            print(f"Делитель {divisor}: встречается {count} раз")
    
    # Возвращаем наиболее вероятные длины ключа
    likely_key_lengths = [divisor for divisor, count in divisor_counts.most_common(key_count)]
    if verbose:
        print(f"\nНаиболее вероятные длины ключа: {likely_key_lengths}")
    
    return likely_key_lengths

def index_of_coincidence(text):
    """
    Вычисление индекса совпадений для текста
    """
    if not text:
        return 0
    
    text = ''.join([ch for ch in text if ch.isalpha()]).upper()
    if len(text) < 2:
        return 0
    
    freq = Counter(text)
    total = len(text)
    
    ic = sum([count * (count - 1) for count in freq.values()]) / (total * (total - 1))
    return ic


def find_key_length_ic(ciphertext, max_key_length=30, key_count=5, verbose=False):
    """
    Определение длины ключа с помощью индекса совпадений
    """
    if verbose:
        print("\n=== ИНДЕКС СОВПАДЕНИЙ ===")
    
    clean_text = ''.join([ch for ch in ciphertext if ch.isalpha()]).upper()
    
    # Ожидаемые значения IC для разных языков
    expected_ic = {
        'russian': 0.0553,
        'english': 0.065
    }
    
    results = []
    
    for key_len in range(1, max_key_length + 1):
        # Разбиваем текст на группы по позициям ключа
        groups = [''] * key_len
        for i, char in enumerate(clean_text):
            groups[i % key_len] += char
        
        # Вычисляем средний IC для всех групп
        group_ics = [index_of_coincidence(group) for group in groups if len(group) > 1]
        if group_ics:
            avg_ic = sum(group_ics) / len(group_ics)
            
            # Вычисляем отклонение от ожидаемого IC (берем русский как основной)
            deviation = abs(avg_ic - expected_ic['russian'])
            
            results.append({
                'key_length': key_len,
                'avg_ic': avg_ic,
                'deviation': deviation
            })
    
    # Сортируем по близости к ожидаемому IC
    results.sort(key=lambda x: x['deviation'])
    
    if verbose:
        print("Топ-10 наиболее вероятных длин ключа по IC:")
        for i, result in enumerate(results[:10]):
            print(f"{i+1}. Длина ключа: {result['key_length']}, IC: {result['avg_ic']:.4f}, "
                  f"отклонение: {result['deviation']:.4f}")
    
    return [result['key_length'] for result in results[:key_count]]

def combined_key_length_analysis(ciphertext, key_count=5, verbose=False):
    """
    Комбинированный анализ длины ключа методами Казиски и IC
    """
    if verbose:
        print("=" * 60)
        print("ОПРЕДЕЛЕНИЕ ДЛИНЫ КЛЮЧА")
        print("=" * 60)
    
    # Метод Казиски
    kasiski_lengths = kasiski_examination(ciphertext, verbose=verbose, key_count=key_count)
    
    # Метод индекса совпадений
    ic_lengths = find_key_length_ic(ciphertext, verbose=verbose, key_count=key_count)
    
    # Комбинируем результаты
    all_suggestions = kasiski_lengths + ic_lengths
    final_suggestions = []
    
    for length in all_suggestions:
        if length not in final_suggestions:
            final_suggestions.append(length)
    
    if verbose:
        print(f"\n=== ИТОГОВЫЕ ПРЕДПОЛОЖЕНИЯ О ДЛИНЕ КЛЮЧА ===")
        print(f"Метод Казиски: {kasiski_lengths}")
        print(f"Метод IC: {ic_lengths}")
        print(f"Объединенный список: {final_suggestions}")
    
    return kasiski_lengths, ic_lengths,final_suggestions


In [ ]:
# Тестируем на криптограммах
cryptograms = {
    'Криптограмма 1': 'ШЩЛБФВЮГЧЙЭМУАПЧЪЫФТЩДЦВЙЖЭЕЮДБЖШФТБЖЮЭИЖЮЛНФВЖБИЩЙФВЖЮГЦЙФТЛНЧЦВФВЖЮГЧЙЭЬУЩФТЛБФВЮГЧЙЭМУАПЧЪЫФТЩДЦВЙЖЭЕЮДБЖШФТБЖЮЭИЖЮЛНФВЖБИЩЙФВЖЮГЦЙФТЛНЧЦВФВЖЮГЧЙЭЬУЩФТЛБФВЮГЧЙЭМУАПЧЪЫФТЩДЦВЙЖЭЕЮДБЖШФТБЖЮЭИЖЮЛНФВЖБИЩЙФВЖЮГЦЙФТЛНЧЦВФВЖЮГЧЙЭЬ',
    'Криптограмма 2': 'ЙСБВЮЕУЙЧТДЖФБНРЮИЧЦХЩЗЙУФВТДЛФЮИЖШОКЧЮИЦФБЩЮЛГЧФЮИЖЮЛГХАФБНТЩЗФЮИЖШОКЧРЮЛГХАФБНТЩЗЙУФВТДЛФЮИЖШОКЧРЮЛГХАЙСБВЮЕУЙЧТДЖФБНРЮИЧЦХЩЗЙУФВТДЛФЮИЖШОКЧРЮЛГХА',
    'Криптограмма 3': 'ЧЛДЙЖФТАПЩЗЙУХМВЖЮЭИФВЛНРБЕЪЫФОКЦЙСГШЭЬУАПЧЙФТДЖЦВЙУФМБНЩЗХЙСГШЭЬУАПЧЙФТДЖЦВЙУФМБНЩЗХЙСГШЭЬУАПЧЙФТДЖЦВЙУФМБНЩЗХЙСГШЭЬУАПЧЙФТДЖЦВЙУФМБНЩЗХ',
    'Криптограмма 4': 'ФЙУГХЩЙЧТЛНРЮИЦВЖШЭЕЬАПБМВЗСЪЫФОКДЖЦЙУГХЩЙЧТЛНРЮИЦВЖШЭЕЬАПБМВЗСЪЫФОКДЖЦЙУГХЩЙЧТЛНРЮИЦВЖШЭЕЬАПБМВЗСЪЫФОКДЖЦЙУГХЩЙЧТЛНРЮИЦВЖШЭЕЬАПБМВЗСЪЫФОКДЖ',
    'Криптограмма 5': 'ЩЗХЙСГШЭЬУАПЧЙФТДЖЦВЙУФМБНРЮИЧЛДЙЖФТАПЩЗЙУХМВЖЮЭИФВЛНРБЕЪЫФОКЦЙСГШЭЬУАПЧЙФТДЖЦВЙУФМБНРЮИЧЛДЙЖФТАПЩЗЙУХМВЖЮЭИФВЛНРБЕЪЫФОКЦЙСГШЭЬУАПЧЙФТДЖЦВЙУФМБНРЮИЧЛДЙЖФТАПЩЗЙУХМВЖЮЭИФВЛНРБЕЪЫФОК'
}

known_keys = {
    'Криптограмма 1': 'криптография',
    'Криптограмма 2': 'шифр',
    'Криптограмма 3': 'стеганография', 
    'Криптограмма 4': 'код',
    'Криптограмма 5': 'алгоритмшмфрования'
}

In [ ]:
# Анализируем все криптограммы
print("АНАЛИЗ КРИПТОГРАММ: ОПРЕДЕЛЕНИЕ ДЛИНЫ КЛЮЧА")
print("=" * 80)

key_length_results = {}

for name, cryptogram in cryptograms.items():
    print(f"\n{'='*60}")
    print(f"АНАЛИЗ: {name}")
    print(f"Известный ключ: '{known_keys[name]}' (длина: {len(known_keys[name])})")
    print(f"Длина криптограммы: {len(cryptogram)} символов")
    print(f"{'='*60}")
    
    kasiski, ic, suggested_lengths = combined_key_length_analysis(cryptogram, key_count=7)
    key_length_results[name] = {
        'suggested': suggested_lengths,
        'actual': len(known_keys[name]),
        'correct': len(known_keys[name]) in suggested_lengths
    }
    
    print(f"✓ Реальная длина ключа: {len(known_keys[name])}")
    print(f"✓ Метод Казиски: {kasiski}")
    print(f"✓ Метод IC: {ic}")
    print(f"✓ Найдена в предложенных: {'ДА' if key_length_results[name]['correct'] else 'НЕТ'}")

In [ ]:
# Применяем методы к нашим зашифрованным текстам из предыдущего задания
print("АНАЛИЗ НАШИХ ЗАШИФРОВАННЫХ ТЕКСТОВ")
print("=" * 80)

# Анализируем текст с коротким ключом
print("\n" + "="*60)
print("АНАЛИЗ: Текст с коротким ключом 'code' (длина: 4)")
print("="*60)

kasiski, ic, short_key_analysis = combined_key_length_analysis(cryptogram, key_count=7)
print(f"✓ Реальная длина ключа: 4")
print(f"✓ Метод Казиски: {kasiski}")
print(f"✓ Метод IC: {ic}")
print(f"✓ Найдена в предложенных: {'ДА' if 4 in short_key_analysis else 'НЕТ'}")

# Анализируем текст с длинным ключом
print("\n" + "="*60)
print("АНАЛИЗ: Текст с длинным ключом 'cryptographysecurity' (длина: 19)")
print("="*60)

kasiski, ic, long_key_analysis = combined_key_length_analysis(encrypted_long, key_count=7)
print(f"✓ Реальная длина ключа: 19")
print(f"✓ Метод Казиски: {kasiski}")
print(f"✓ Метод IC: {ic}")
print(f"✓ Найдена в предложенных: {'ДА' if 19 in long_key_analysis else 'НЕТ'}")

# Задание 3 (Криптоанализ - взлом ключа)

In [ ]:
RUSSIAN_FREQUENCIES = {
    'о': 0.1097, 'е': 0.0845, 'а': 0.0801, 'и': 0.0735, 'н': 0.0670,
    'т': 0.0626, 'с': 0.0547, 'р': 0.0473, 'в': 0.0454, 'л': 0.0440,
    'к': 0.0349, 'м': 0.0321, 'д': 0.0298, 'п': 0.0281, 'у': 0.0262,
    'я': 0.0201, 'ы': 0.0190, 'ь': 0.0174, 'г': 0.0170, 'з': 0.0165,
    'б': 0.0159, 'ч': 0.0144, 'й': 0.0121, 'х': 0.0097, 'ж': 0.0094,
    'ш': 0.0073, 'ю': 0.0064, 'ц': 0.0048, 'щ': 0.0036, 'э': 0.0032,
    'ф': 0.0026, 'ъ': 0.0004, 'ё': 0.0004
}

In [ ]:
def caesar_shift_russian(text, shift):
    """
    Применение шифра Цезаря с заданным сдвигом для русского алфавита
    """
    result = []
    
    for char in text:
        if char.lower() in RUSSIAN_ALPHABET:
            char_lower = char.lower()
            idx = RUSSIAN_ALPHABET.index(char_lower)
            new_idx = (idx - shift) % RUSSIAN_ALPHABET_SIZE  # Вычитаем для расшифровки
            new_char = RUSSIAN_ALPHABET[new_idx]
            
            # Сохраняем регистр
            if char.isupper():
                new_char = new_char.upper()
            result.append(new_char)
        else:
            result.append(char)
    
    return ''.join(result)

def find_best_shift_russian(text):
    """
    Нахождение лучшего сдвига для русского текста с помощью метода хи-квадрат
    """
    clean_text = ''.join([ch.lower() for ch in text if ch.lower() in RUSSIAN_ALPHABET])
    if not clean_text:
        return 0, float('inf')
    
    best_shift = 0
    best_chi_square = float('inf')
    
    # Пробуем все возможные сдвиги
    for shift in range(RUSSIAN_ALPHABET_SIZE):
        # Расшифровываем текст с этим сдвигом
        decrypted = caesar_shift_russian(clean_text, shift)
        
        # Подсчитываем частоты
        freq = Counter(decrypted)
        total_chars = len(decrypted)
        
        # Вычисляем хи-квадрат
        chi_square = 0
        for char, expected_prob in RUSSIAN_FREQUENCIES.items():
            observed = freq.get(char, 0)
            expected = expected_prob * total_chars
            if expected > 0:
                chi_square += (observed - expected) ** 2 / expected
        
        if chi_square < best_chi_square:
            best_chi_square = chi_square
            best_shift = shift
    
    return best_shift, best_chi_square

In [ ]:
def break_vigenere_key_russian(ciphertext, key_length, verbose=True):
    """
    Взлом ключа шифра Виженера для русского текста при известной длине ключа
    """
    if verbose:
        print(f"=== ВЗЛОМ КЛЮЧА (длина: {key_length}) ===")
    
    # Разбиваем криптограмму на группы
    groups = [''] * key_length
    for i, char in enumerate(ciphertext):
        if char.lower() in RUSSIAN_ALPHABET:
            groups[i % key_length] += char
    
    if verbose:
        print(f"\nАнализ {key_length} групп символов:")
        for i, group in enumerate(groups):
            print(f"  Группа {i}: {len(group)} символов")
    
    # Находим сдвиг для каждой позиции ключа
    key_shifts = []
    key_chars = []
    
    for i, group in enumerate(groups):
        if group:
            shift, chi_square = find_best_shift_russian(group)
            key_shifts.append(shift)
            key_char = RUSSIAN_ALPHABET[shift]
            key_chars.append(key_char)
            
            if verbose:
                print(f"Позиция {i}: сдвиг {shift} → буква '{key_char}' (χ²={chi_square:.2f})")
        else:
            key_shifts.append(0)
            key_chars.append('?')
    
    recovered_key = ''.join(key_chars)
    if verbose:
        print(f"\nВосстановленный ключ: '{recovered_key}'")
    
    return recovered_key, key_shifts

def decrypt_vigenere_russian(ciphertext, key):
    """
    Расшифровка русского текста с использованием ключа
    """
    result = []
    key = key.lower()
    key_length = len(key)
    key_index = 0
    
    for char in ciphertext:
        if char.lower() in RUSSIAN_ALPHABET:
            # Получаем сдвиг из ключа
            key_char = key[key_index % key_length]
            shift = RUSSIAN_ALPHABET.index(key_char)
            
            # Применяем обратный сдвиг
            char_lower = char.lower()
            idx = RUSSIAN_ALPHABET.index(char_lower)
            new_idx = (idx - shift) % RUSSIAN_ALPHABET_SIZE
            new_char = RUSSIAN_ALPHABET[new_idx]
            
            # Сохраняем регистр
            if char.isupper():
                new_char = new_char.upper()
            
            result.append(new_char)
            key_index += 1
        else:
            result.append(char)
    
    return ''.join(result)

In [ ]:
def evaluate_decryption_quality_russian(text):
    """
    Оценка качества расшифровки русского текста с помощью хи-квадрат
    """
    clean_text = ''.join([ch.lower() for ch in text if ch.isalpha() and ch.lower() in RUSSIAN_ALPHABET])
    if not clean_text:
        return float('inf')
    
    freq = Counter(clean_text)
    total_chars = len(clean_text)
    
    chi_square = 0
    for char, expected_prob in RUSSIAN_FREQUENCIES.items():
        observed = freq.get(char, 0)
        expected = expected_prob * total_chars
        if expected > 0:
            chi_square += (observed - expected) ** 2 / expected
    
    return chi_square

def complete_cryptanalysis_russian(ciphertext, max_key_length=30, verbose=True):
    """
    Полный криптоанализ шифра Виженера для русского текста
    """
    print("=" * 80)
    print("ПОЛНЫЙ КРИПТОАНАЛИЗ ШИФРА ВИЖЕНЕРА (РУССКИЙ ТЕКСТ)")
    print("=" * 80)
    
    # Шаг 1: Определяем длину ключа
    if verbose:
        print("\n1. ОПРЕДЕЛЕНИЕ ДЛИНЫ КЛЮЧА")
    
    _, _, key_lengths = combined_key_length_analysis(ciphertext, verbose=verbose, key_count=8)
    
    if not key_lengths:
        print("Не удалось определить длину ключа")
        return None, None
    
    # Шаг 2: Пробуем все возможные длины ключа
    best_key = None
    best_decryption = None
    best_score = float('inf')
    
    if verbose:
        print(f"\n2. ПЕРЕБОР ВОЗМОЖНЫХ ДЛИН КЛЮЧА: {key_lengths}")
    
    for key_len in key_lengths:
        if verbose:
            print(f"\n--- Проверка длины ключа: {key_len} ---")
        
        # Взламываем ключ
        recovered_key, shifts = break_vigenere_key_russian(ciphertext, key_len, verbose=verbose)
        
        # Расшифровываем текст
        decrypted_text = decrypt_vigenere_russian(ciphertext, recovered_key)
        
        # Оцениваем качество расшифровки
        score = evaluate_decryption_quality_russian(decrypted_text)
        
        if verbose:
            sample = decrypted_text[:100] + "..." if len(decrypted_text) > 100 else decrypted_text
            print(f"Пример расшифровки: {sample}")
            print(f"Оценка качества: {score:.4f}")
        
        if score < best_score:
            best_score = score
            best_key = recovered_key
            best_decryption = decrypted_text
                
    return best_key, best_decryption

In [ ]:
# Тестируем на всех криптограммах
print("\n" + "="*80)
print("ТЕСТИРОВАНИЕ НА ВСЕХ КРИПТОГРАММАХ")
print("="*80)

results = []

for name, ciphertext in cryptograms.items():
    print(f"\n{'='*60}")
    print(f"АНАЛИЗ: {name}")
    print(f"Известный ключ: '{known_keys[name]}'")
    print(f"Длина криптограммы: {len(ciphertext)}")
    print(f"{'='*60}")
    
    # Выполняем криптоанализ
    recovered_key, decrypted_text = complete_cryptanalysis_russian(ciphertext, verbose=False)
    
    if recovered_key:
        is_correct = (recovered_key == known_keys[name])
        results.append({
            'Криптограмма': name,
            'Известный ключ': known_keys[name],
            'Восстановленный ключ': recovered_key,
            'Совпадение': '✓' if is_correct else '✗',
            'Длина текста': len(decrypted_text)
        })
        
        print(f"Восстановленный ключ: '{recovered_key}'")
        print(f"Результат: {'✓ УСПЕХ' if is_correct else '✗ ОШИБКА'}")
        
        # Показываем маленький пример
        sample = decrypted_text[:50] + "..." if len(decrypted_text) > 50 else decrypted_text
        print(f"Пример: {sample}")
    else:
        print("Не удалось восстановить ключ")
        results.append({
            'Криптограмма': name,
            'Известный ключ': known_keys[name],
            'Восстановленный ключ': 'НЕ УДАЛОСЬ',
            'Совпадение': '✗',
            'Длина текста': 0
        })

# Сводная таблица результатов
print("\n" + "="*80)
print("СВОДНАЯ ТАБЛИЦА РЕЗУЛЬТАТОВ")
print("="*80)

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

# Статистика успешности
success_count = sum(1 for r in results if r['Совпадение'] == '✓')
print(f"\nУспешных взломов: {success_count}/{len(results)} ({success_count/len(results)*100:.1f}%)")